# Final project

**Course:** COSC 073 - Computational Photography  
**Term:** Spring 2026  
**Author:** Kasuti Makau

In [25]:
import cv2
import os
from pathlib import Path

In [26]:
DATA_DIR = Path("data")
OUT_DIR = DATA_DIR / "frames"
VIDEO_EXTS = {".mov", ".MOV"}
MAX_FRAMES = 50

In [27]:
# sorted list of (trial_dir, video_file) pairs
videos = []
for trial_dir in sorted(DATA_DIR.iterdir()):
    if not trial_dir.is_dir() or trial_dir.name == "frames":
        continue
    for video_file in sorted(trial_dir.iterdir()):
        if video_file.suffix in VIDEO_EXTS:
            videos.append((trial_dir, video_file))

print(f"Found {len(videos)} videos across {len(set(t for t,_ in videos))} trials\n")


Found 25 videos across 5 trials



In [28]:
# function to extract frames from the video
def extract_frames(video_path: Path, out_dir: Path, max_frames: int = MAX_FRAMES) -> dict:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {}

    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    step = max(1, total // max_frames)

    out_dir.mkdir(parents=True, exist_ok=True)

    saved, idx = 0, 0
    while True:
        ret, frame = cap.read()
        if not ret or saved >= max_frames:
            break
        if idx % step == 0:
            cv2.imwrite(str(out_dir / f"frame_{idx:05d}.jpg"), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
            saved += 1
        idx += 1

    cap.release()
    return {"fps": fps, "total": total, "saved": saved, "w": width, "h": height}

print(f"{'Trial':<12} {'Video':<15} {'FPS':>6} {'Total':>7} {'Saved':>6} {'Resolution'}")
print("-" * 60)

for trial_dir, video_file in videos:
    out_subdir = OUT_DIR / trial_dir.name / video_file.stem.lower()
    info = extract_frames(video_file, out_subdir)
    if info:
        print(
            f"{trial_dir.name:<12} {video_file.stem:<15} "
            f"{info['fps']:>6.1f} {info['total']:>7} {info['saved']:>6}  "
            f"{info['w']}x{info['h']}"
        )


Trial        Video              FPS   Total  Saved Resolution
------------------------------------------------------------
trial1       1200hz            30.0     615     50  1920x1080
trial1       200hz             30.0     618     50  1920x1080
trial1       440hz             30.0     610     50  1920x1080
trial1       880hz             30.0     618     50  1920x1080
trial1       silence           30.0     612     50  1920x1080
trial2       1200hz            30.0     611     50  1920x1080
trial2       200hz             30.0     625     50  1920x1080
trial2       440hz             30.0     613     50  1920x1080
trial2       880hz             30.0     612     50  1920x1080
trial2       silence           30.0     623     50  1920x1080
trial3       1200hz            30.0     610     50  1920x1080
trial3       200hz             30.0     611     50  1920x1080
trial3       440hz             30.0     613     50  1920x1080
trial3       880hz             30.0     666     50  1920x1080
trial3   